# Fase 0 — Auditoría de `get_features_lexical()`

**Objetivo (según el enunciado del taller):** antes de agregar características nuevas, verificar que las ~29 características que ya entrega `get_features_lexical()` realmente funcionan. Para cada una se reporta:

- **(a)** ¿es constante en el corpus?
- **(b)** ¿es un simple proxy de la longitud del texto (correlación con el número de tokens)?
- **(c)** ¿deja de activarse por el orden del preprocesamiento (acentos, emojis, hashtags)?

Y al final: corregir la característica, o justificar por qué se excluye.


In [6]:
%pip install nltk
%pip install spacy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import sys
sys.path.insert(0, '..')  # ajustar si el notebook no está en /notebooks

import re
import numpy as np
import pandas as pd
from nltk import TweetTokenizer

from logic.text_processing import TextProcessing
from logic.feature_extraction import FeatureExtraction

tp = TextProcessing(lang='es')
fe = FeatureExtraction(lang='es')
tt = TweetTokenizer()

df = pd.read_csv('../data/tass/tass2018_es_train.csv')
texts = df['content'].dropna().tolist()
print(f"N tweets en train: {len(texts)}")


N tweets en train: 1008


## Paso 1 — Construir la matriz de características sobre TODO el corpus

Se corre `get_features_lexical()` sobre cada tweet (después de pasar por el mismo `transformer()` que se usa en el pipeline real), y se guarda en paralelo el número de tokens de cada tweet (con el mismo `TweetTokenizer`, para que el conteo sea comparable con el que usa la propia función internamente).


In [8]:
NOMBRES_FEATURES = [
    'weighted_position', 'weighted_normalized', 'label_mention', 'label_url', 'label_hashtag',
    'label_emoji', 'label_retweets', 'lexical_diversity', 'label_word',
    'first_person_singular', 'second_person_singular', 'third_person_singular',
    'first_person_plurar', 'second_person_plurar', 'third_person_plurar',
    'avg_word', 'kur_word', 'skew_word',
    'adverb_neg', 'adverb_time', 'adverb_place', 'adverb_mode', 'adverb_cant', 'adverb_all',
    'adjetives_neg', 'adjetives_pos', 'who_general', 'who_male', 'who_female'
]

filas = []
n_tokens_lista = []
saltados = 0

for t in texts:
    proc = tp.transformer(t)
    if not proc:
        saltados += 1
        continue
    vec = fe.get_features_lexical(proc)
    if vec is None or len(vec) != len(NOMBRES_FEATURES):
        saltados += 1
        continue
    tokens = tt.tokenize(proc)
    filas.append(vec)
    n_tokens_lista.append(len(tokens))

matriz = np.array(filas)
n_tok = np.array(n_tokens_lista)
print(f"Filas usables: {matriz.shape[0]} | Saltadas: {saltados}")


Filas usables: 1008 | Saltadas: 0


## Paso 2 — Chequeos (a) constante y (b) proxy de longitud

- **(a)** Se calcula la desviación estándar de cada columna. Una desviación ≈ 0 significa que el valor nunca cambia entre tweets: el clasificador no puede aprender nada de ahí (equivalente a lo que hace `sklearn.feature_selection.VarianceThreshold`).
- **(b)** Se calcula la correlación de Pearson de cada columna contra el número de tokens del tweet. Una correlación fuerte (`|r| > 0.6` como umbral de alerta) indica que la característica está capturando principalmente la longitud del texto, no su contenido.


In [9]:
UMBRAL_CORR = 0.6

resultados = []
for i, nombre in enumerate(NOMBRES_FEATURES):
    col = matriz[:, i]
    std = np.std(col)
    es_constante = std < 1e-9
    corr = np.corrcoef(col, n_tok)[0, 1] if not es_constante else np.nan
    pct_ceros = np.mean(col == 0) * 100
    alerta = es_constante or (not np.isnan(corr) and abs(corr) > UMBRAL_CORR)
    resultados.append({
        'feature': nombre,
        'constante': es_constante,
        'std': round(std, 4),
        'corr_con_longitud': round(corr, 4) if not np.isnan(corr) else np.nan,
        'pct_ceros': round(pct_ceros, 1),
        'ALERTA': alerta
    })

tabla_auditoria = pd.DataFrame(resultados)
tabla_auditoria.sort_values('ALERTA', ascending=False)


,feature,constante,std,corr_con_longitud,pct_ceros,ALERTA
0,weighted_position,False,0.7406,0.7169,0.0,True
5,label_emoji,True,0.0000,NaN,100.0,True
1,weighted_normalized,False,2.7152,0.9842,0.0,True
7,lexical_diversity,False,0.1001,-0.8262,0.0,True
8,label_word,False,5.9969,0.9870,0.0,True
4,label_hashtag,True,0.0000,NaN,100.0,True
25,adjetives_pos,False,0.4924,0.1429,78.3,False
26,who_general,False,0.0991,0.0015,99.0,False
17,skew_word,False,0.5516,0.2049,1.1,False
24,adjetives_neg,False,0.3780,0.0496,87.0,False


## Paso 3 — Chequeo (c): ¿se rompe por el orden del preprocesamiento?

Aquí no basta la estadística — hay que **trazar el pipeline paso a paso** sobre ejemplos reales que contengan el elemento en cuestión (hashtag, emoji), imprimiendo el resultado después de cada transformación, hasta encontrar exactamente en qué paso se pierde o se corrompe la señal que `get_features_lexical()` espera encontrar.

### 3.1 — Caso hashtag


In [10]:
tweet_hashtag = next(t for t in texts if '#' in t)
print("Tweet crudo:", repr(tweet_hashtag))

t1 = TextProcessing.proper_encoding(tweet_hashtag).lower()
print("1) proper_encoding + lower:", repr(t1))

t2 = re.sub("#([A-Za-z0-9_]{1,40})", '[HASTAG]', t1)
print("2) regex de hashtag (tal como está escrito en text_processing.py):", repr(t2))

t3 = TextProcessing.remove_patterns(t2)
print("3) remove_patterns (quita corchetes y pasa a minúscula):", repr(t3))

print()
print("Token literal que SÍ se genera:", "'hastag'" if 'hastag' in t3.split() else "no aparece 'hastag'")
print("¿Aparece el token 'hashtag' que busca get_features_lexical()?:", 'hashtag' in t3.split())


Tweet crudo: 'Se ha terminado #Rio2016 Lamentablemente no arriendo las ganancias al pueblo brasileño por la penuria que les espera \nSuerte y solidaridad'
1) proper_encoding + lower: 'se ha terminado #rio2016 lamentablemente no arriendo las ganancias al pueblo brasileno por la penuria que les espera \nsuerte y solidaridad'
2) regex de hashtag (tal como está escrito en text_processing.py): 'se ha terminado [HASTAG] lamentablemente no arriendo las ganancias al pueblo brasileno por la penuria que les espera \nsuerte y solidaridad'
3) remove_patterns (quita corchetes y pasa a minúscula): 'se ha terminado hastag lamentablemente no arriendo las ganancias al pueblo brasileno por la penuria que les espera \nsuerte y solidaridad'

Token literal que SÍ se genera: 'hastag'
¿Aparece el token 'hashtag' que busca get_features_lexical()?: False


### 3.2 — Caso emoji

In [11]:
emoji_pattern = re.compile("[\U0001F300-\U0001FAFF\U00002600-\U000027BF\U0001F1E6-\U0001F1FF]+")
tweet_emoji = next(t for t in texts if emoji_pattern.search(t))
print("Tweet crudo:", repr(tweet_emoji))

t1 = TextProcessing.proper_encoding(tweet_emoji)
print("1) proper_encoding:", repr(t1))
print("   -> ¿el emoji sigue presente después de este paso?:", bool(emoji_pattern.search(t1)))

t1_lower = t1.lower()
t2 = re.sub("[\U0001f000-\U000e007f]", '[EMOJI]', t1_lower)
print("2) regex de emoji (nunca tiene nada que reemplazar, porque ya se perdió en el paso 1):", repr(t2))

texto_final = TextProcessing.transformer(tweet_emoji)
print("\nResultado final de transformer():", repr(texto_final))
print("¿Aparece el token 'emoji'?:", 'emoji' in texto_final.split())


Tweet crudo: '@Sakura_Abril Ow \nBueno, no pasa nada, cuando puedas confirmarlo, estoy aquí 😊\nY si no pudieras de cosplay pero sí a la expo, +'
1) proper_encoding: '@Sakura_Abril Ow \nBueno, no pasa nada, cuando puedas confirmarlo, estoy aqui \nY si no pudieras de cosplay pero si a la expo, +'
   -> ¿el emoji sigue presente después de este paso?: False
2) regex de emoji (nunca tiene nada que reemplazar, porque ya se perdió en el paso 1): '@sakura_abril ow \nbueno, no pasa nada, cuando puedas confirmarlo, estoy aqui \ny si no pudieras de cosplay pero si a la expo, +'

Resultado final de transformer(): 'mention ow bueno no pasa nada cuando puedas confirmarlo estoy aqui y si no pudieras de cosplay pero si a la expo'
¿Aparece el token 'emoji'?: False


**Causa raíz:** `proper_encoding()` usa `text.encode('ascii', 'ignore')` para quitar tildes, pero esto también descarta *cualquier* carácter no-ASCII, incluyendo los emojis — que se destruyen **antes** de que el regex de `[EMOJI]` en `transformer()` tenga oportunidad de detectarlos. Por diseño (orden de las operaciones), `label_emoji` nunca puede activarse, sin importar cuántos emojis tenga el tweet original.


## Paso 4 — Pruebas unitarias (evidencia reproducible, no solo prints)

Se formalizan los hallazgos como pruebas con `assert`, para que cualquiera pueda re-ejecutar el notebook y confirmar (o refutar, si se corrige el código más adelante) los mismos resultados.


In [12]:
def test_hashtag_bug_presente():
    resultado = TextProcessing.transformer("Esto es un tema #importante")
    tokens = resultado.split()
    assert 'hastag' in tokens, "Se esperaba encontrar el token roto 'hastag'"
    assert 'hashtag' not in tokens, "El token correcto 'hashtag' no debería aparecer (bug confirmado)"
    print("OK: bug de hashtag confirmado (token real = 'hastag', no 'hashtag')")

def test_emoji_bug_presente():
    resultado = TextProcessing.transformer("Que alegria!! 😊 🎉")
    tokens = resultado.split()
    assert 'emoji' not in tokens, "El token 'emoji' no debería aparecer (bug confirmado)"
    print("OK: bug de emoji confirmado (los emojis se pierden en proper_encoding, nunca llegan a 'emoji')")

def test_weighted_normalized_es_proxy_de_longitud():
    # Con la formula teorica (n+1)/2, para dos tweets de igual longitud (sin tokens repetidos)
    # el valor de weighted_normalized deberia depender solo de n, no del contenido
    t_a = "casa perro gato pajaro arbol flor"      # 6 palabras distintas
    t_b = "rojo verde negro blanco morado violeta"  # 6 palabras distintas
    vec_a = fe.get_features_lexical(t_a)
    vec_b = fe.get_features_lexical(t_b)
    idx = NOMBRES_FEATURES.index('weighted_normalized')
    assert abs(vec_a[idx] - vec_b[idx]) < 1e-6, "weighted_normalized deberia ser igual para dos textos de igual longitud, sin importar el contenido"
    print(f"OK: weighted_normalized es igual ({vec_a[idx]:.4f}) para dos textos distintos de la misma longitud -> confirma proxy de longitud")

test_hashtag_bug_presente()
test_emoji_bug_presente()
test_weighted_normalized_es_proxy_de_longitud()


OK: bug de hashtag confirmado (token real = 'hastag', no 'hashtag')
OK: bug de emoji confirmado (los emojis se pierden en proper_encoding, nunca llegan a 'emoji')
OK: weighted_normalized es igual (3.5000) para dos textos distintos de la misma longitud -> confirma proxy de longitud


## Paso 5 — Tabla resumen de la Fase 0

| Feature | Veredicto | Evidencia | Causa raíz |
|---|---|---|---|
| `label_hashtag` | Constante en 0 | 100% de ceros en el corpus completo (n=1008) | Typo: `text_processing.py` genera `[HASTAG]` (sin la "h" de "hash"), `feature_extraction.py` busca `'hashtag'` |
| `label_emoji` | Constante en 0 | 100% de ceros en el corpus completo | `proper_encoding()` destruye los emojis con `encode('ascii','ignore')` antes de que el regex de `[EMOJI]` pueda actuar |
| `weighted_position` | Proxy de longitud | Correlación 0.72 con n° de tokens | Fórmula basada en `.index()`, que además falla con tokens repetidos (usa la posición del primer duplicado, no la real) |
| `weighted_normalized` | Proxy de longitud casi perfecto | Correlación 0.98 con n° de tokens | Fórmula `Σ(1+i)/n` se reduce algebraicamente a `(n+1)/2` |
| `label_word` | Prácticamente = n° de tokens | Correlación 0.99 con n° de tokens | Por construcción: total de tokens menos tags, que casi nunca se activan |
| `lexical_diversity` | Dependiente de la longitud | Correlación -0.83 con n° de tokens | Propiedad conocida del TTR (type-token ratio) en la literatura de PLN |
| `adverb_all` | Redundante (no rompe los 3 chequeos oficiales, pero es 100% colineal) | Es la suma exacta de las otras 5 categorías de adverbio | Combinación lineal determinística de columnas ya existentes |
| Resto (22 features) | Sin alertas | `\|r\|` < 0.6 y no constantes | — |

**Nota:** la decisión de corregir vs. excluir cada una (y su justificación formal) es el siguiente paso de la Fase 0, pendiente de definir antes de avanzar a la Fase 1.
